# 06 — VLM ↔ LRP Image Concept Mapping

Aligns two sets of image-level concepts per sample:
- **VLM concepts** (`referred_image_concepts`): what the model self-reported as visually important
- **LRP concepts** (`lrp_visual_features`): words projected from the LRP heatmap via GPT-4o (from `04`)

An OpenAI LLM finds the best **one-to-one semantic mapping** between the two sets.  
Concepts with no reasonable counterpart are left unmatched. Each concept is used at most once.

Outputs → `data/vlm_heatmap_mapping/<mode>/mapping.json`

In [1]:
CONFIG = {
    "vlm_explanations_dir": "./data/vlm_explanations",
    "output_dir": "./data/vlm_heatmap_mapping",
    "visual_modes": ["image", "image+text"],
    "language": "uk",
    "gpt_model": "gpt-4o",
    "max_concurrent": 20,
}

In [6]:
import asyncio
import json
import os
from pathlib import Path

from json_repair import repair_json
from openai import AsyncOpenAI

OUT_DIR = Path(CONFIG["output_dir"])
OUT_DIR.mkdir(parents=True, exist_ok=True)
EXPL_DIR = Path(CONFIG["vlm_explanations_dir"])
api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise RuntimeError("OPENAI_API_KEY environment variable must be set.")

client = AsyncOpenAI(api_key=api_key)

In [7]:
LANG_NOTE = {
    "uk": "The concepts are in Ukrainian. Match based on meaning, not exact wording.",
    "en": "The concepts are in English. Match based on meaning, not exact wording.",
}

MAPPING_PROMPT = """\
You are aligning two lists of visual concepts extracted from the same propaganda meme image.

List A — VLM self-reported concepts (what the vision-language model said it focused on):
{vlm_list}

List B — LRP heatmap concepts (visual elements highlighted by gradient attribution, described by a VLM):
{lrp_list}

{lang_note}

Task: find the best one-to-one semantic mapping between concepts in A and B.

Rules:
- Each concept may be used at most once.
- Only match a pair if both concepts clearly refer to the same visual element (same person, object, or region).
- Do NOT force matches — if a concept has no clear counterpart, leave it unmatched.
- Prefer precision: a smaller set of confident matches is better than many uncertain ones.

Return ONLY valid JSON (no markdown, no explanation):
{{"matches": [["concept from A", "concept from B"], ...],
  "unmatched_vlm": ["concept from A with no match", ...],
  "unmatched_lrp": ["concept from B with no match", ...]}}
"""


async def map_concepts(sample_id, vlm_concepts, lrp_concepts, semaphore):
    """Ask LLM to produce a one-to-one mapping between two concept sets."""
    vlm_concepts = list(vlm_concepts or [])
    lrp_concepts = list(lrp_concepts or [])

    # Skip LLM call when one or both lists are empty
    if not vlm_concepts and not lrp_concepts:
        return {"id": sample_id, "matches": [], "unmatched_vlm": [], "unmatched_lrp": [], "skipped": True}
    if not vlm_concepts:
        return {"id": sample_id, "matches": [], "unmatched_vlm": [], "unmatched_lrp": lrp_concepts, "skipped": True}
    if not lrp_concepts:
        return {"id": sample_id, "matches": [], "unmatched_vlm": vlm_concepts, "unmatched_lrp": [], "skipped": True}

    def fmt(concepts):
        return "\n".join(f"  {i+1}. {c}" for i, c in enumerate(concepts))

    lang_note = LANG_NOTE.get(CONFIG["language"], "")
    prompt = MAPPING_PROMPT.format(
        vlm_list=fmt(vlm_concepts),
        lrp_list=fmt(lrp_concepts),
        lang_note=lang_note,
    )

    async with semaphore:
        try:
            resp = await client.chat.completions.create(
                model=CONFIG["gpt_model"],
                messages=[
                    {"role": "system", "content": "You are a precise semantic alignment assistant. Return only valid JSON."},
                    {"role": "user", "content": prompt},
                ],
                temperature=0,
                max_completion_tokens=500,
            )
            raw = resp.choices[0].message.content.strip().replace("```json", "").replace("```", "").strip()
            parsed = json.loads(repair_json(raw))

            # Validate: matched concepts must appear in the original lists (LLM may hallucinate)
            vlm_set = set(vlm_concepts)
            lrp_set = set(lrp_concepts)
            valid_matches = [
                [a, b] for a, b in parsed.get("matches", [])
                if a in vlm_set and b in lrp_set
            ]

            return {
                "id": sample_id,
                "matches": valid_matches,
                "unmatched_vlm": parsed.get("unmatched_vlm", []),
                "unmatched_lrp": parsed.get("unmatched_lrp", []),
                "raw_response": raw,
            }
        except Exception as e:
            print(f"  Mapping error [{sample_id}]: {e}")
            return {
                "id": sample_id,
                "matches": [],
                "unmatched_vlm": vlm_concepts,
                "unmatched_lrp": lrp_concepts,
                "error": str(e),
            }

In [8]:
async def run_mapping(mode):
    mode_key = mode.replace("+", "_")
    expl_file = EXPL_DIR / mode_key / "explanations.json"
    out_dir = OUT_DIR / mode_key
    out_dir.mkdir(parents=True, exist_ok=True)

    records = json.load(open(expl_file))
    semaphore = asyncio.Semaphore(CONFIG["max_concurrent"])

    tasks = [
        map_concepts(
            r["id"],
            r.get("referred_image_concepts", []),
            r.get("lrp_visual_features", []),
            semaphore,
        )
        for r in records
    ]

    print(f"[{mode}] Mapping concepts for {len(tasks)} samples...")
    results = await asyncio.gather(*tasks)

    # Attach per-sample metrics
    id_to_expl = {r["id"]: r for r in records}
    for res in results:
        expl = id_to_expl[res["id"]]
        n_vlm = len(expl.get("referred_image_concepts", []))
        n_lrp = len(expl.get("lrp_visual_features", []))
        n_matches = len(res["matches"])
        denom = max(n_vlm, n_lrp)
        res["n_vlm"] = n_vlm
        res["n_lrp"] = n_lrp
        res["n_matches"] = n_matches
        # match_rate: matched pairs / max possible (bounded by the larger set)
        res["match_rate"] = n_matches / denom if denom > 0 else 0.0

    out_file = out_dir / "mapping.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(list(results), f, ensure_ascii=False, indent=2)

    matched = sum(1 for r in results if r["n_matches"] > 0)
    avg_rate = sum(r["match_rate"] for r in results) / len(results) if results else 0
    skipped = sum(1 for r in results if r.get("skipped"))
    print(f"[{mode}] {matched}/{len(results)} samples with ≥1 match | "
          f"avg match_rate={avg_rate:.3f} | {skipped} skipped (empty list)")
    print(f"[{mode}] Saved → {out_file}")
    return results


all_results = {}
for mode in CONFIG["visual_modes"]:
    all_results[mode] = await run_mapping(mode)

[image] Mapping concepts for 158 samples...
[image] 88/158 samples with ≥1 match | avg match_rate=0.207 | 4 skipped (empty list)
[image] Saved → data/vlm_heatmap_mapping/image/mapping.json
[image+text] Mapping concepts for 158 samples...
[image+text] 115/158 samples with ≥1 match | avg match_rate=0.316 | 5 skipped (empty list)
[image+text] Saved → data/vlm_heatmap_mapping/image_text/mapping.json


In [10]:
# ── Summary ───────────────────────────────────────────────────────────────────
for mode, results in all_results.items():
    print(f"\n=== {mode} ===")
    n = len(results)
    rates = [r["match_rate"] for r in results]
    print(f"  Samples:            {n}")
    print(f"  With ≥1 match:      {sum(1 for r in results if r['n_matches'] > 0)}")
    print(f"  Avg match_rate:     {sum(rates)/n:.3f}")
    print(f"  Avg n_matches:      {sum(r['n_matches'] for r in results)/n:.2f}")
    print(f"  Avg n_vlm concepts: {sum(r['n_vlm'] for r in results)/n:.2f}")
    print(f"  Avg n_lrp concepts: {sum(r['n_lrp'] for r in results)/n:.2f}")

    # Show 3 examples with matches
    print("\n  Examples:")
    examples = [r for r in results][:10]
    for ex in examples:
        print(f"\n  [{ex['id']}]")
        for vlm_c, lrp_c in ex["matches"]:
            print(f"    '{vlm_c}'  →  '{lrp_c}'")
        if ex["unmatched_vlm"]:
            print(f"    unmatched VLM: {ex['unmatched_vlm']}")
        if ex["unmatched_lrp"]:
            print(f"    unmatched LRP: {ex['unmatched_lrp']}")


=== image ===
  Samples:            158
  With ≥1 match:      88
  Avg match_rate:     0.207
  Avg n_matches:      0.68
  Avg n_vlm concepts: 3.18
  Avg n_lrp concepts: 2.04

  Examples:

  [847_batch_2]
    'владимир путін'  →  'володимир путін'
    'кім чен ын'  →  'кім чен ин'
    'обама'  →  'барак обама'
    unmatched VLM: ['гітлер', 'прапор']

  [766_batch_2]
    'хілларі'  →  'жінка з розкритим ротом і світлим волоссям'
    'прапор'  →  'емодзі та прапор у нижній частині зображення'
    unmatched VLM: ['обама', 'берні', 'відкриті очі']
    unmatched LRP: ['молода жінка з великими очима і зібраним волоссям', 'чоловік що кричить у мікрофон', 'жінка з молотком що кричить']

  [723_batch_2]
    'байден'  →  'джо байден'
    unmatched VLM: ['фейк-переможець', 'сукня']

  [742_batch_2]
    unmatched VLM: ['вона: «фрукт чи колір', 'сміється', 'попросили по літерах «orange']
    unmatched LRP: ['жінка']

  [767_batch_2]
    unmatched VLM: ['прапор', 'піднятий кулак', 'гроби']
    unmat